In [1]:
import os, glob, json, time
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"; os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"; os.environ["HF_HUB_DISABLE_XET"] = "1"
import numpy as np, pandas as pd, torch, torch.nn as nn, lightgbm as lgb
from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoModel, AutoTokenizer
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score

def find(name, many=False):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"{name} not found: check the attached datasets (aes_dataset must be version 1)"
    return hits if many else hits[0]

conf = json.load(open(find("aes_blend_config.json")))
feature_cols, W = conf["feature_cols"], conf["deberta_weight"]
fold_files = find("aes_fold*.pt", many=True)
print("DeBERTa fold models found:", len(fold_files), "| blend weight:", W)

asap = (pd.read_csv(find("asap_prepared.csv"))
        .merge(pd.read_csv(find("asap_features.csv")), on="essay_id")
        .merge(pd.read_csv(find("asap_languagetool.csv")), on="essay_id")
        .merge(pd.read_csv(find("asap_cola.csv")), on="essay_id"))
print(asap.shape, "| missing feature columns:", [c for c in feature_cols if c not in asap.columns])

# ---------- DeBERTa scores (average of the 5 fold models) ----------
path = snapshot_download("microsoft/deberta-v3-base", allow_patterns=["*.json", "spm.model"])
tok = AutoTokenizer.from_pretrained(path)
assert (tok.pad_token_id, tok.cls_token_id, tok.sep_token_id) == (0, 1, 2)
CLS, SEP, MAXLEN = 1, 2, 512

def wrap(ids):
    room = MAXLEN - 2
    if len(ids) > room:
        head = int(room * 0.75)
        ids = ids[:head] + ids[len(ids) - (room - head):]
    return [CLS] + ids + [SEP]

enc = [wrap(i) for i in tok(asap["text"].tolist(), add_special_tokens=False)["input_ids"]]

class Scorer(nn.Module):
    def __init__(self):
        super().__init__()
        c = AutoConfig.from_pretrained(path)
        self.backbone = AutoModel.from_config(c)
        self.head = nn.Linear(c.hidden_size, 1)
    def forward(self, ids, att):
        h = self.backbone(input_ids=ids, attention_mask=att).last_hidden_state.float()
        m = att.unsqueeze(-1).float()
        return self.head((h * m).sum(1) / m.sum(1).clamp(min=1))

def batch_tensors(idx):
    L = max(len(enc[i]) for i in idx)
    ids = torch.zeros((len(idx), L), dtype=torch.long); att = torch.zeros_like(ids)
    for r, i in enumerate(idx):
        n = len(enc[i]); ids[r, :n] = torch.tensor(enc[i]); att[r, :n] = 1
    return ids.cuda(), att.cuda()

order = np.argsort([len(e) for e in enc])
preds = np.zeros((len(fold_files), len(enc)), dtype=np.float32)
for k, f in enumerate(fold_files):
    t0 = time.time()
    model = Scorer().cuda().eval()
    model.load_state_dict({kk: v.float() for kk, v in torch.load(f, map_location="cpu").items()})
    for s in range(0, len(order), 64):
        sel = order[s:s + 64]
        ids, att = batch_tensors(sel)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            preds[k, sel] = model(ids, att).float().cpu().numpy()[:, 0]
    print(f"fold model {k}: {(time.time() - t0) / 60:.1f} min")
    del model; torch.cuda.empty_cache()

asap["pred_deberta"] = preds.mean(0)
booster = lgb.Booster(model_file=find("lgbm_aes_features.txt"))
asap["pred_lgbm"] = booster.predict(asap[feature_cols])
asap["pred"] = W * asap["pred_deberta"] + (1 - W) * asap["pred_lgbm"]
asap[["essay_id", "essay_set", "task_type", "domain1_score", "score_scaled", "pred_deberta", "pred_lgbm", "pred"]
     ].to_csv("/kaggle/working/asap_predictions.csv", index=False)

# ---------- Evaluation per essay set ----------
SET_RANGES = {1: (2, 12), 2: (1, 6), 3: (0, 3), 4: (0, 3), 5: (0, 4), 6: (0, 4), 7: (0, 30), 8: (0, 60)}
def calibrated_qwk(s, g):
    lo, hi = SET_RANGES[s]
    z = (g["pred"] - g["pred"].mean()) / (g["pred"].std() + 1e-9)
    mapped = np.clip(np.rint(g["domain1_score"].mean() + z * g["domain1_score"].std()), lo, hi).astype(int)
    return cohen_kappa_score(g["domain1_score"], mapped, weights="quadratic")

rows = []
for s, g in asap.groupby("essay_set"):
    rows.append({"set": s, "type": g["task_type"].iloc[0], "n": len(g),
                 "blend": spearmanr(g["pred"], g["domain1_score"])[0],
                 "deberta": spearmanr(g["pred_deberta"], g["domain1_score"])[0],
                 "lgbm": spearmanr(g["pred_lgbm"], g["domain1_score"])[0],
                 "length only": spearmanr(g["n_words"], g["domain1_score"])[0],
                 "QWK (calibrated)": calibrated_qwk(s, g)})
res = pd.DataFrame(rows).set_index("set")
print("\nPer essay set (Spearman correlation with the human score):")
print(res.round(3).to_string())
print("\nAverage by task type:")
print(res.groupby("type")[["blend", "deberta", "lgbm", "length only", "QWK (calibrated)"]].mean().round(3).to_string())

DeBERTa fold models found: 5 | blend weight: 0.65
(12972, 71) | missing feature columns: []


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

[transformers] The tokenizer you are loading from '/root/.cache/huggingface/hub/models--microsoft--deberta-v3-base/snapshots/8ccc9b6f36199bec6961081d44eb72fb3f7353f3' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


fold model 0: 1.8 min
fold model 1: 1.9 min
fold model 2: 1.9 min
fold model 3: 1.9 min
fold model 4: 1.9 min

Per essay set (Spearman correlation with the human score):
             type     n  blend  deberta   lgbm  length only  QWK (calibrated)
set                                                                          
1           essay  1783  0.758    0.743  0.730        0.782             0.746
2           essay  1800  0.732    0.715  0.723        0.664             0.689
3    short_answer  1724  0.596    0.589  0.432        0.718             0.554
4    short_answer  1768  0.616    0.600  0.483        0.759             0.590
5    short_answer  1805  0.709    0.696  0.597        0.832             0.649
6    short_answer  1800  0.713    0.716  0.571        0.683             0.664
7           essay  1569  0.449    0.315  0.501        0.681             0.446
8           essay   723  0.586    0.460  0.673        0.438             0.590

Average by task type:
              blend  debert